# 광명시 스쿨존 - 누락 컬럼 추가
스쿨존별 반경 300m 내 **무인교통단속카메라** 및 **생활안전CCTV** 개수 산출

In [1]:
import pandas as pd
import numpy as np

## 1. 데이터 로드

In [2]:
# 스쿨존 메인 데이터
df = pd.read_csv('5_gwangmyung_final_dataset.csv', encoding='cp949')

# 생활안전 CCTV 데이터
df_cctv = pd.read_csv('광명시_보호구역내cctv정보.csv', encoding='cp949')

# 무인교통단속카메라 데이터
df_cam = pd.read_csv('광명시_보호구역내교통단속카메라정보.csv', encoding='cp949')

print('스쿨존 데이터:', df.shape)
print('생활안전CCTV:', df_cctv.shape)
print('교통단속카메라:', df_cam.shape)

스쿨존 데이터: (51, 24)
생활안전CCTV: (98, 15)
교통단속카메라: (45, 22)


In [3]:
print('스쿨존 컬럼:', df.columns.tolist())
print()
print('CCTV 컬럼:', df_cctv.columns.tolist())
print()
print('카메라 컬럼:', df_cam.columns.tolist())

스쿨존 컬럼: ['시설물명', '위도', '경도', '과속방지턱', '도로적색표면', '무단횡단방지펜스', '보호구역표지판', '신호등', '옐로카펫', '횡단보도', '보호구역 도로폭', '발생건수', '사상자수', '사망및중상자수', '사망자수', '중상자수', '경상자수', '동', '총인구수', '0~4세', '5~9세', '10~14세', '어린이 총인구', '어린이 비율(%)']

CCTV 컬럼: ['SFZN_ID', 'INST_NM', 'RDNMADR', 'LNMADR', 'INST_PURPS', 'CCTV_CNT', 'CCTV_PIXEL', 'PTGRF_INFO', 'CSTDY_DAY', 'INST_YYMM', 'PH_NUMBER', '경도', '위도', 'REF_DATE', 'ID_PRT']

카메라 컬럼: ['보호구역아이디', '관리번호', '시도명', '시군구명', '도로종류', '도로노선번호', '도로명', '도로노선방향', '소재지도로명주소', '소재지지번주소', '경도', '위도', '설치장소', '단속구분', '제한속도', '단속구간위치구분', '과속단속구간길이', '보호구역구분', '설치년도', '관리기관명', '관리기관전화번호', '데이터기준일자']


## 2. 위경도 컬럼 추출

In [4]:
# 스쿨존: col[1]=위도, col[2]=경도
# CCTV:   col[11]=경도, col[12]=위도
# 카메라: col[10]=경도, col[11]=위도
sz_lat_col   = df.columns[1]
sz_lon_col   = df.columns[2]

cctv_lon_col = df_cctv.columns[11]
cctv_lat_col = df_cctv.columns[12]
cctv_cnt_col = 'CCTV_CNT'

cam_lon_col  = df_cam.columns[10]
cam_lat_col  = df_cam.columns[11]

print('스쿨존 위도:', sz_lat_col, '/ 경도:', sz_lon_col)
print('CCTV   위도:', cctv_lat_col, '/ 경도:', cctv_lon_col)
print('카메라 위도:', cam_lat_col,  '/ 경도:', cam_lon_col)

스쿨존 위도: 위도 / 경도: 경도
CCTV   위도: 위도 / 경도: 경도
카메라 위도: 위도 / 경도: 경도


## 3. Haversine 거리 계산 함수

In [5]:
def haversine_distance(lat1, lon1, lat2_arr, lon2_arr):
    """기준점(lat1, lon1)과 배열 간의 Haversine 거리를 미터(m) 단위로 반환"""
    R = 6371000  # 지구 반지름 (m)
    lat1_r = np.radians(lat1)
    lon1_r = np.radians(lon1)
    lat2_r = np.radians(lat2_arr)
    lon2_r = np.radians(lon2_arr)

    dlat = lat2_r - lat1_r
    dlon = lon2_r - lon1_r

    a = np.sin(dlat / 2)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

## 4. 반경 300m 내 CCTV / 카메라 개수 산출

In [6]:
RADIUS = 300  # 기준 반경 (m)

# CCTV 좌표 배열 (NaN 제거)
cctv_valid = df_cctv[[cctv_lat_col, cctv_lon_col, cctv_cnt_col]].dropna(subset=[cctv_lat_col, cctv_lon_col])
cctv_lats  = cctv_valid[cctv_lat_col].values
cctv_lons  = cctv_valid[cctv_lon_col].values
cctv_cnts  = cctv_valid[cctv_cnt_col].fillna(1).values

# 카메라 좌표 배열 (NaN 제거)
cam_valid = df_cam[[cam_lat_col, cam_lon_col]].dropna(subset=[cam_lat_col, cam_lon_col])
cam_lats  = cam_valid[cam_lat_col].values
cam_lons  = cam_valid[cam_lon_col].values

cctv_counts = []
cam_counts  = []

for _, row in df.iterrows():
    sz_lat = row[sz_lat_col]
    sz_lon = row[sz_lon_col]

    # 생활안전CCTV: 반경 내 CCTV_CNT 합산
    dist_cctv = haversine_distance(sz_lat, sz_lon, cctv_lats, cctv_lons)
    cctv_counts.append(int(cctv_cnts[dist_cctv <= RADIUS].sum()))

    # 무인교통단속카메라: 반경 내 설치 지점 수
    dist_cam = haversine_distance(sz_lat, sz_lon, cam_lats, cam_lons)
    cam_counts.append(int((dist_cam <= RADIUS).sum()))

df['생활안전CCTV']       = cctv_counts
df['무인교통단속카메라'] = cam_counts

print('컬럼 추가 완료!')
print(df[['생활안전CCTV', '무인교통단속카메라']].describe())

컬럼 추가 완료!
        생활안전CCTV  무인교통단속카메라
count  51.000000  51.000000
mean    5.823529   1.549020
std     6.208723   1.064213
min     0.000000   0.000000
25%     2.000000   1.000000
50%     4.000000   2.000000
75%     6.000000   2.000000
max    27.000000   4.000000


## 5. 결과 확인

In [7]:
name_col = df.columns[0]  # 시설물명
df[[name_col, sz_lat_col, sz_lon_col, '생활안전CCTV', '무인교통단속카메라']].head(10)

,시설물명,위도,경도,생활안전CCTV,무인교통단속카메라
0,빛가온초등학교,37.413690,126.882026,11,2
1,빛가온유치원,37.416952,126.888265,0,0
2,광명생명숲어린이집,37.432309,126.877817,2,0
3,큰별어린이집,37.436146,126.877327,0,1
4,충현초등학교,37.432490,126.884627,2,2
5,트인아이유치원,37.434968,126.886029,2,1
6,아란유치원,37.437262,126.879689,2,2
7,서면초등학교,37.438660,126.879511,2,2
8,광명푸른숲어린이집,37.477792,126.850856,23,3
9,예림유치원,37.468770,126.852730,12,2


In [8]:

name_col = df.columns[0]  # 시설물명

no_cctv = df[df['생활안전CCTV'] == 0]
no_cam  = df[df['무인교통단속카메라'] == 0]

print(f'=== 반경 300m 내 생활안전CCTV 없는 스쿨존: {len(no_cctv)}개 ===')
print(no_cctv[[name_col, '생활안전CCTV', '무인교통단속카메라']].to_string(index=False))

print()
print(f'=== 반경 300m 내 무인교통단속카메라 없는 스쿨존: {len(no_cam)}개 ===')
print(no_cam[[name_col, '생활안전CCTV', '무인교통단속카메라']].to_string(index=False))


=== 반경 300m 내 생활안전CCTV 없는 스쿨존: 3개 ===
  시설물명  생활안전CCTV  무인교통단속카메라
빛가온유치원         0          0
큰별어린이집         0          1
 세교유치원         0          0

=== 반경 300m 내 무인교통단속카메라 없는 스쿨존: 9개 ===
     시설물명  생활안전CCTV  무인교통단속카메라
   빛가온유치원         0          0
광명생명숲어린이집         2          0
    세교유치원         0          0
   안서초등학교         3          0
    산들유치원         2          0
   구름산유치원         7          0
  구름산어린이집         1          0
    예원유치원         2          0
    삼흥유치원         2          0


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 26 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   시설물명       51 non-null     object 
 1   위도         51 non-null     float64
 2   경도         51 non-null     float64
 3   과속방지턱      51 non-null     int64  
 4   도로적색표면     51 non-null     int64  
 5   무단횡단방지펜스   51 non-null     int64  
 6   보호구역표지판    51 non-null     int64  
 7   신호등        51 non-null     int64  
 8   옐로카펫       51 non-null     int64  
 9   횡단보도       51 non-null     int64  
 10  보호구역 도로폭   51 non-null     int64  
 11  발생건수       51 non-null     int64  
 12  사상자수       51 non-null     int64  
 13  사망및중상자수    51 non-null     int64  
 14  사망자수       51 non-null     int64  
 15  중상자수       51 non-null     int64  
 16  경상자수       51 non-null     int64  
 17  동          51 non-null     object 
 18  총인구수       51 non-null     int64  
 19  0~4세       51 non-null     int64  
 20  5~9세       5

## 6. 저장

In [11]:
output_path = '../../data/5_gwangmyung_final_dataset.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {output_path}')
print('최종 shape:', df.shape)
print('최종 컬럼:', df.columns.tolist())

저장 완료: ../../data/5_gwangmyung_final_dataset.csv
최종 shape: (51, 26)
최종 컬럼: ['시설물명', '위도', '경도', '과속방지턱', '도로적색표면', '무단횡단방지펜스', '보호구역표지판', '신호등', '옐로카펫', '횡단보도', '보호구역 도로폭', '발생건수', '사상자수', '사망및중상자수', '사망자수', '중상자수', '경상자수', '동', '총인구수', '0~4세', '5~9세', '10~14세', '어린이 총인구', '어린이 비율(%)', '생활안전CCTV', '무인교통단속카메라']


## 3) 도로안전표지 컬럼 추가

스쿨존 중심 좌표 기준 반경 300m 이내의 `도로안전표지현황(제공표준).csv` 표지 수를 계산해 `final_df`에 추가하고 CSV를 업데이트합니다.

In [9]:
# 도로안전표지 데이터 로드
sign_df = pd.read_csv('./도로안전표지현황(제공표준).csv', encoding='cp949')
final_df = pd.read_csv('../../data/5_gwangmyung_final_dataset_renew.csv', encoding='cp949')
print(f"도로안전표지 전체 행 수: {len(sign_df):,}")

# 스쿨존 중심 기준 반경 300m 이내 도로안전표지 수 계산 (Haversine 거리)
def count_in_radius(zone_lat, zone_lon, poi_lats, poi_lons, radius_m=300):
    R = 6371000  # 지구 반지름 (m)
    phi1 = np.radians(zone_lat)
    phi2 = np.radians(poi_lats)
    dphi = phi2 - phi1
    dlambda = np.radians(poi_lons - zone_lon)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    dists = 2 * R * np.arcsin(np.sqrt(a))
    return int((dists <= radius_m).sum())

sign_lats = sign_df['위도'].values
sign_lons = sign_df['경도'].values

# final_df에 도로안전표지 컬럼 추가 (이미 있으면 덮어쓰기)
final_df['도로안전표지'] = final_df.apply(
    lambda row: count_in_radius(row['위도'], row['경도'], sign_lats, sign_lons),
    axis=1
)

print("\n[스쿨존별 도로안전표지 수]")
print(final_df[['시설물명', '도로안전표지']].to_string())
print(f"\n합계: {final_df['도로안전표지'].sum()}  |  평균: {final_df['도로안전표지'].mean():.2f}  |  최대: {final_df['도로안전표지'].max()}")

도로안전표지 전체 행 수: 44,055

[스쿨존별 도로안전표지 수]
         시설물명  도로안전표지
0     빛가온초등학교       0
1      빛가온유치원       0
2   광명생명숲어린이집       0
3      큰별어린이집       0
4      충현초등학교       0
5     트인아이유치원       0
6       아란유치원       0
7      서면초등학교       0
8   광명푸른숲어린이집       0
9       예림유치원       0
10     예크어린이집       0
11   엄지창의어린이집       0
12      녹야유치원       0
13     광명초등학교       0
14    광명남초등학교       0
15    광명서초등학교       0
16     광문초등학교       0
17     광일초등학교       0
18      세교유치원       0
19     온신초등학교       0
20     안서초등학교       0
21      산들유치원       0
22    하안남초등학교       0
23      예솔유치원       0
24     구름산유치원       0
25     가림초등학교       0
26     연서초등학교       0
27     파랑새유치원       1
28    구름산초등학교       0
29     소하초등학교       0
30     홍익어린이집       0
31    마리아어린이집       0
32     광덕초등학교       1
33     안현초등학교       0
34    구름산어린이집       0
35    하안북초등학교       0
36      가림유치원       0
37     하안초등학교       1
38      예원유치원       1
39      예지유치원       0
40      예지유치원       0
41    광명북초등학교       0
42    광명북초등학교  

In [10]:
# '도로안전표지' 컬럼을 '과속방지턱' 바로 앞에 위치시키고 CSV 저장
cols = list(final_df.columns)
if '도로안전표지' in cols:
    cols.remove('도로안전표지')
insert_idx = cols.index('과속방지턱')
cols.insert(insert_idx, '도로안전표지')
final_df = final_df[cols]

DATASET_PATH = '../../data/5_gwangmyung_final_dataset.csv'
final_df.to_csv(DATASET_PATH, index=False, encoding='utf-8-sig')
print(f"저장 완료: {DATASET_PATH}")
print(f"컬럼 수: {final_df.shape[1]}  (기존 26 → {final_df.shape[1]})")
final_df.head(3)

저장 완료: ../../data/5_gwangmyung_final_dataset.csv
컬럼 수: 27  (기존 26 → 27)


,시설물명,위도,경도,도로안전표지,과속방지턱,도로적색표면,무단횡단방지펜스,무인교통단속카메라,보호구역표지판,생활안전CCTV,...,사망자수,중상자수,경상자수,동,총인구수,0~4세,5~9세,10~14세,어린이 총인구,어린이 비율(%)
0,빛가온초등학교,37.413690,126.882026,0,10,13,19,2,14,8,...,0,0,0,일직동,21148,851,979,1264,3094,14.630225
1,빛가온유치원,37.416952,126.888265,0,1,4,12,0,9,0,...,0,0,0,일직동,21148,851,979,1264,3094,14.630225
2,광명생명숲어린이집,37.432309,126.877817,0,2,0,0,0,8,2,...,0,0,0,소하동,54145,1089,1725,2722,5536,10.224397


# 광명시 보호표지판 컬럼 재추가